In [1]:
import urllib
import re

import numpy as np
from collections import Counter
import random

In [5]:
class Word2Vec:
    def __init__(self, vocab_size, emb_dim, lr=0.01, seed=42):
        np.random.seed(seed)
        self.W_in = np.random.randn(vocab_size, emb_dim) * 0.1
        self.W_out = np.random.randn(vocab_size, emb_dim) * 0.1
        self.lr = lr
        self.cache = {}

    # helper function
    def _sigmoid(self, x):
        # Numerical stability: clip values
        x = np.clip(x, -15, 15)
        return 1 / (1 + np.exp(-x))
    
    # helper function
    def _sample_negative(self, vocab_size, true_context, num_negative):
        negatives = []
        while len(negatives) < num_negative:
            neg = random.randint(1, vocab_size - 1)   # avoid PAD=0 if you want
            if neg != true_context:
                negatives.append(neg)
        return negatives

    def forward(self, center_id, context_id, negative_ids):
        # 1. Get vectors
        v_c = self.W_in[center_id]
        u_pos = self.W_out[context_id]
        u_negs = self.W_out[negative_ids] # Matrix: (num_neg, emb_dim)

        # 2. Compute scores & probabilities
        pos_score = np.dot(u_pos, v_c)
        pos_prob = self._sigmoid(pos_score)

        neg_scores = np.dot(u_negs, v_c) # Array: (num_neg,)
        neg_probs = self._sigmoid(neg_scores)

        # 3. Store in cache for backward pass
        self.cache = {
            'v_c': v_c, 'u_pos': u_pos, 'u_negs': u_negs,
            'pos_prob': pos_prob, 'neg_probs': neg_probs,
            'center_id': center_id, 'context_id': context_id, 'negative_ids': negative_ids
        }

        # Calculate Loss: -log(p_pos) - sum(log(1 - p_neg)) (e.g. loss(positive_score) - sum(loss(negative_scores)))
        loss = -np.log(pos_prob + 1e-12) - np.sum(np.log(1 - neg_probs + 1e-12))
        return loss

    def backward(self):
        # Retrieve from cache
        v_c = self.cache['v_c']
        u_pos = self.cache['u_pos']
        u_negs = self.cache['u_negs']
        pos_prob = self.cache['pos_prob']
        neg_probs = self.cache['neg_probs']

        # 1. Error terms (prediction - label)
        # Label for positive is 1, Label for negative is 0
        grad_z_pos = pos_prob - 1
        grad_z_negs = neg_probs # (neg_probs - 0)

        # 2. Gradients for weights
        # Gradient for center word: dL/dv_c
        grad_v_c = grad_z_pos * u_pos + np.dot(grad_z_negs, u_negs)

        # Gradient for positive context word: dL/du_pos
        grad_u_pos = grad_z_pos * v_c

        # Gradient for negative context words: dL/du_neg
        grad_u_negs = np.outer(grad_z_negs, v_c)

        return grad_v_c, grad_u_pos, grad_u_negs
    
    def train(self, x_train, y_train, x_val, y_val, vocab_size, epochs, num_negative):
        for epoch in range(epochs):
            # --- Training ---
            total_train_loss = 0
            indices = np.arange(len(x_train))
            np.random.shuffle(indices)

            for idx in indices:
                c_id, context_id = x_train[idx], y_train[idx]
                neg_ids = self._sample_negative(vocab_size, context_id, num_negative)

                total_train_loss += self.forward(c_id, context_id, neg_ids)
                g_vc, g_up, g_un = self.backward()

                # Update
                self.W_in[c_id] -= self.lr * g_vc
                self.W_out[context_id] -= self.lr * g_up
                self.W_out[neg_ids] -= self.lr * g_un

            # --- Validation (nur Forward) ---
            total_val_loss = 0
            for i in range(len(x_val)):
                c_id, context_id = x_val[i], y_val[i]
                neg_ids = self._sample_negative(vocab_size, context_id, num_negative)
                total_val_loss += self.forward(c_id, context_id, neg_ids)

            print(f"Epoch {epoch+1}/{epochs} - Loss: {total_train_loss/len(x_train):.4f} - Val-Loss: {total_val_loss/len(x_val):.4f}")

In [6]:
# creates vocab, mappings and translates word tokens to int tokens
def build_vocab(tokens):
    pad_token = "<PAD>"
    unk_token = "<UNK>"

    word_counts = Counter(tokens)

    vocab = [pad_token, unk_token] + sorted(word_counts.keys())
    w2i = {w: i for i, w in enumerate(vocab)}
    i2w = {i: w for w, i in w2i.items()}

    pad_id = w2i[pad_token]
    unk_id = w2i[unk_token]
    vocab_size = len(vocab)

    # turn token stream into ids
    token_ids = [w2i.get(w, unk_id) for w in tokens]
    
    return token_ids, w2i, i2w, vocab_size


def build_dataset(token_ids, window_size):
    window_size = 2

    x_train = []
    y_train = []

    for i in range(window_size, len(token_ids) - window_size):
        center = token_ids[i]

        x_train.append(center)
        y_train.append(token_ids[i - 2])

        x_train.append(center)
        y_train.append(token_ids[i - 1])

        x_train.append(center)
        y_train.append(token_ids[i + 1])

        x_train.append(center)
        y_train.append(token_ids[i + 2])

    x_train = np.array(x_train, dtype=np.int32)
    y_train = np.array(y_train, dtype=np.int32)

    return x_train, y_train


def load_alice():
    url = "https://www.gutenberg.org/files/11/11-0.txt"
    response = urllib.request.urlopen(url)
    raw_text = response.read().decode('utf8')
    
    # Einfaches Cleaning: Nur Buchstaben, alles kleingeschrieben
    tokens = re.findall(r'\b\w+\b', raw_text.lower())
    
    # Optional: Häufige Stoppwörter oder sehr seltene Wörter entfernen, 
    # um das Vokabular klein zu halten
    return tokens

In [ ]:
# 1. Load data
data = load_alice()
print(f"Gesamtanzahl Wörter: {len(data)}")
print(f"Einzigartige Wörter: {len(set(data))}")

print(data[:1000])

# 2. Translate words to integeres, create mappings and calculate vocab_size + create centers and contexts in seperate lists
token_ids, w2i, i2w, vocab_size = build_vocab(data)
x, y = build_dataset(token_ids, window_size=2)

# split data into 90/10
split = int(len(x) * 0.9)
x_train, y_train = x[:split], y[:split]
x_val, y_val = x[split:], y[split:]
model = Word2Vec(vocab_size, emb_dim=10, lr=0.05)

model.train(x_train, y_train, x_val, y_val, vocab_size, epochs=20, num_negative=4)

# final test if meaningful features were learned
def cosine_sim(word1, word2):
    if word1 not in w2i or word2 not in w2i: return "N/A"
    v1, v2 = model.W_in[w2i[word1]], model.W_in[w2i[word2]]
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("\nFinal Cosine Similarities:")
test_pairs = [("alice", "rabbit"), ("alice", "she"), ("queen", "king"), ("cat", "dog")]
for w1, w2 in test_pairs:
    print(f"{w1} <-> {w2}: {cosine_sim(w1, w2)}")

Gesamtanzahl Wörter: 27455
Einzigartige Wörter: 2695
['start', 'of', 'the', 'project', 'gutenberg', 'ebook', '11', 'illustration', 'alice', 's', 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', 'the', 'millennium', 'fulcrum', 'edition', '3', '0', 'contents', 'chapter', 'i', 'down', 'the', 'rabbit', 'hole', 'chapter', 'ii', 'the', 'pool', 'of', 'tears', 'chapter', 'iii', 'a', 'caucus', 'race', 'and', 'a', 'long', 'tale', 'chapter', 'iv', 'the', 'rabbit', 'sends', 'in', 'a', 'little', 'bill', 'chapter', 'v', 'advice', 'from', 'a', 'caterpillar', 'chapter', 'vi', 'pig', 'and', 'pepper', 'chapter', 'vii', 'a', 'mad', 'tea', 'party', 'chapter', 'viii', 'the', 'queen', 's', 'croquet', 'ground', 'chapter', 'ix', 'the', 'mock', 'turtle', 's', 'story', 'chapter', 'x', 'the', 'lobster', 'quadrille', 'chapter', 'xi', 'who', 'stole', 'the', 'tarts', 'chapter', 'xii', 'alice', 's', 'evidence', 'chapter', 'i', 'down', 'the', 'rabbit', 'hole', 'alice', 'was', 'beginning', 'to', 'get', 'ver

In [9]:
data = load_alice()
print(f"Gesamtanzahl Wörter: {len(data)}")
print(f"Einzigartige Wörter: {len(set(data))}")

print(data[:1000])

Gesamtanzahl Wörter: 27455
Einzigartige Wörter: 2695
['start', 'of', 'the', 'project', 'gutenberg', 'ebook', '11', 'illustration', 'alice', 's', 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', 'the', 'millennium', 'fulcrum', 'edition', '3', '0', 'contents', 'chapter', 'i', 'down', 'the', 'rabbit', 'hole', 'chapter', 'ii', 'the', 'pool', 'of', 'tears', 'chapter', 'iii', 'a', 'caucus', 'race', 'and', 'a', 'long', 'tale', 'chapter', 'iv', 'the', 'rabbit', 'sends', 'in', 'a', 'little', 'bill', 'chapter', 'v', 'advice', 'from', 'a', 'caterpillar', 'chapter', 'vi', 'pig', 'and', 'pepper', 'chapter', 'vii', 'a', 'mad', 'tea', 'party', 'chapter', 'viii', 'the', 'queen', 's', 'croquet', 'ground', 'chapter', 'ix', 'the', 'mock', 'turtle', 's', 'story', 'chapter', 'x', 'the', 'lobster', 'quadrille', 'chapter', 'xi', 'who', 'stole', 'the', 'tarts', 'chapter', 'xii', 'alice', 's', 'evidence', 'chapter', 'i', 'down', 'the', 'rabbit', 'hole', 'alice', 'was', 'beginning', 'to', 'get', 'ver